# From structured grid to Tecplot
## Данный скрипт выполняет обработку данных из structured grid в формат Tecplot

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# папка с файлами
folder = "data/Combustion"

files = {
    "U": "combustionU.txt",
    "V": "combustionV.txt",
    "T": "combustionT.txt",
    "O2": "combustionO2.txt",
    "CO2": "combustionCO2.txt",
    "CH4": "combustionCH4.txt",
    "CO": "combustionCO.txt",
}

data_vars = {}

# читаем первый файл, чтобы получить X и Y
first = list(files.values())[0]
df = pd.read_csv(os.path.join(folder, first), sep=r"\s+", header=None)

x = df.iloc[0, 1:].values.astype(float)
y = df.iloc[1:, 0].values.astype(float)

NX = len(x)
NY = len(y)

# читаем все поля
for var_name, filename in files.items():
    df = pd.read_csv(os.path.join(folder, filename), sep=r"\s+", header=None)
    Z = df.iloc[1:, 1:].values.astype(float)
    data_vars[var_name] = Z

# запись в Tecplot
with open("combined.dat", "w") as f:
    # VARIABLES
    var_list = ['"X"', '"Y"'] + [f'"{v}"' for v in files.keys()]
    f.write("VARIABLES = " + ", ".join(var_list) + "\n")

    # ZONE
    f.write(f"ZONE I={NX}, J={NY}, F=POINT\n")

    # запись данных
    for j in range(NY):
        for i in range(NX):
            line = [x[i], y[j]]

            for var in files.keys():
                line.append(data_vars[var][j, i])

            f.write(" ".join(map(str, line)) + "\n")

In [10]:
T = pd.read_csv(r"C:\Users\гыук\PycharmProjects\project\data\Combustion\Silva_T.txt", sep="\t", header=1)
CH4 = pd.read_csv(r"C:\Users\гыук\PycharmProjects\project\data\Combustion\Silva_CH4.txt", sep="\t", header=1)
CO2 = pd.read_csv(r"C:\Users\гыук\PycharmProjects\project\data\Combustion\Silva_CO2.txt", sep="\t", header=1)
CO = pd.read_csv(r"C:\Users\гыук\PycharmProjects\project\data\Combustion\Silva_CO.txt", sep="\t", header=1)

X = T['X']

df = pd.DataFrame({
    'X': X,
    'T': T['T'],
    'CH4': np.interp(X, CH4['X'], CH4['CH4']),
    'CO2': np.interp(X, CO2['X'], CO2['CO2']),
    'CO': np.interp(X, CO['X'], CO['CO'])
})

df.to_csv("tecplot.dat", index=False, sep=' ')

In [9]:
import glob
import os

folder = "data/Combustion"

files = glob.glob(os.path.join(folder, "SECTION_*_*.txt"))

In [10]:
data = {}

for file in files:
    filename = os.path.basename(file).replace(".txt", "")

    # SECTION_1_O2 → ["SECTION", "1", "O2"]
    parts = filename.split("_")

    if len(parts) != 3:
        continue

    _, sec, var = parts

    sec_name = f"section{sec.lower()}"
    var_name = var.lower()

    if sec_name not in data:
        data[sec_name] = {}

    df = pd.read_csv(file, sep="\t")

    # стандартизируем
    df.columns = ["X", "Y", "VALUE"]
    df = df.dropna().sort_values("Y")

    data[sec_name][var_name] = df

In [11]:
data

{'section1': {'co2':         X       Y    VALUE
  0   312.0    0.00  0.00380
  1   312.0    7.68  0.00345
  2   312.0   13.01  0.00411
  3   312.0   18.86  0.00580
  4   312.0   27.80  0.01057
  5   312.0   34.92  0.01599
  6   312.0   41.62  0.02207
  7   312.0   48.79  0.02953
  8   312.0   54.58  0.03595
  9   312.0   63.07  0.04676
  10  312.0   71.16  0.05695
  11  312.0   79.64  0.06711
  12  312.0   88.13  0.07657
  13  312.0   95.76  0.08334
  14  312.0  105.11  0.08876
  15  312.0  113.19  0.09384
  16  312.0  121.68  0.09722
  17  312.0  130.62  0.09961
  18  312.0  140.03  0.10195
  19  312.0  149.38  0.10364
  20  312.0  158.32  0.10533
  21  312.0  168.18  0.10638
  22  312.0  177.53  0.10668
  23  312.0  186.48  0.10772
  24  312.0  196.34  0.10872
  25  312.0  206.61  0.10872
  26  312.0  215.09  0.10941
  27  312.0  224.50  0.11006
  28  312.0  234.36  0.11076
  29  312.0  243.71  0.11110
  30  312.0  250.00  0.11176,
  'o2':       X       Y    VALUE
  0   312    0.00  

In [12]:
for sec in data:
    print(sec, list(data[sec].keys()))

section1 ['co2', 'o2', 't']
section2 ['co2', 'o2', 't']
section3 ['co2', 'o2', 't']


In [15]:
y_common = np.linspace(0, 250, 200)

interp_data = {}

for sec in data:
    interp_data[sec] = {}

    for var in data[sec]:
        df = data[sec][var]

        interp_data[sec][var] = np.interp(
            y_common,
            df["Y"],
            df["VALUE"]
        )

In [16]:
interp_data

{'section1': {'co2': array([0.0038    , 0.00374275, 0.0036855 , 0.00362824, 0.00357099,
         0.00351374, 0.00345649, 0.00358794, 0.0037435 , 0.00389906,
         0.00405463, 0.00434374, 0.00470666, 0.00506959, 0.00543252,
         0.00579544, 0.00646188, 0.00713218, 0.00780247, 0.00847277,
         0.00914307, 0.00981337, 0.01048367, 0.01140315, 0.01235948,
         0.0133158 , 0.01427213, 0.01522846, 0.0162222 , 0.01736223,
         0.01850226, 0.01964229, 0.02078232, 0.02192234, 0.0232078 ,
         0.02451489, 0.02582199, 0.02712908, 0.02843617, 0.02975728,
         0.03115025, 0.03254323, 0.0339362 , 0.03532918, 0.03683668,
         0.03843625, 0.04003583, 0.04163541, 0.04323498, 0.04483456,
         0.04643413, 0.04802002, 0.04960241, 0.0511848 , 0.05276718,
         0.05434957, 0.05593196, 0.0574868 , 0.05899197, 0.06049714,
         0.0620023 , 0.06350747, 0.06501264, 0.06651781, 0.06795907,
         0.06935889, 0.0707587 , 0.07215851, 0.07355833, 0.07495814,
         0.0763

In [17]:
sections = sorted(data.keys())

X_vals = [data[sec]["o2"]["X"].iloc[0] for sec in sections]

In [18]:
X_vals

[np.int64(312), np.int64(912), np.float64(1312.0)]

In [19]:
# порядок секций
sections = sorted(data.keys())  # ['section1', 'section2', 'section3']

# X координаты (312, 912, 1312)
X_vals = [data[sec]["o2"]["X"].iloc[0] for sec in sections]

NX = len(sections)
NY = len(y_common)

# создаём массивы
O2  = np.zeros((NY, NX))
T   = np.zeros((NY, NX))
CO2 = np.zeros((NY, NX))

for i, sec in enumerate(sections):
    O2[:, i]  = interp_data[sec]["o2"]
    T[:, i]   = interp_data[sec]["t"]
    CO2[:, i] = interp_data[sec]["co2"]

# --- запись в Tecplot ---
with open("radial_silva.dat", "w") as f:
    f.write('VARIABLES = "X", "Y", "O2", "T", "CO2"\n')
    f.write(f'ZONE I={NX}, J={NY}, DATAPACKING=POINT\n')

    for j in range(NY):
        for i in range(NX):
            f.write(
                f"{X_vals[i]} {y_common[j]} "
                f"{O2[j,i]} {T[j,i]} {CO2[j,i]}\n"
            )